In [10]:
# Cell 1: Imports & device
import os, time, math, csv
from pathlib import Path
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader

import glob
from torch.utils.data import DataLoader
import torchvision.transforms as T
import torchvision.utils as vutils

# skimage for color conversions & SSIM
from skimage.color import lab2rgb, rgb2lab
from skimage.metrics import structural_similarity as ssim_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [11]:
# Cell 2: Hyperparameters
batch_size = 16
num_epochs = 100
pretrain_epochs = 10

# High-quality, stable LRs
learning_rate_g = 1e-4   # conservative generator LR
learning_rate_d = 2e-4   # conservative, slower discriminator
betas = (0.5, 0.999)

# label smoothing ranges & D input noise
real_label_smooth_min = 0.8
real_label_smooth_max = 1.0
fake_label_max = 0.2
d_input_noise_std = 0.05

# scheduler
decay_start_epoch = int(num_epochs * 0.5)

# logging & saving
save_dir = Path("/home/mmanani/mtechpracticals/semester-4/ChatGPT")
save_dir.mkdir(parents=True, exist_ok=True)
log_csv = save_dir / "training_log.csv"
sample_dir = save_dir / "samples"
sample_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = save_dir / "checkpoint_latest.pth"

# misc
sample_every_batches = 1000
lambda_pixel = 50.0   # pixel (L1) weight; tune if needed
print("Hyperparams set.")


Hyperparams set.


In [12]:
# Cell 3: Dataset loader scanning all subfolders
root_image_dir = "/home/mmanani/ImageColourizationDataSet/ILSVRC/Data/DET/"

# collect image files recursively
img_extensions = ("*.jpg","*.jpeg","*.png","*.bmp")
paths = []
for ext in ["jpg", "jpeg", "png", "JPG", "JPEG", "PNG"]:
    paths.extend(glob.glob(f"{root_image_dir}/**/*.{ext}", recursive=True))

print(f"Found {len(paths)} image files (first 3):", paths[:3])

if len(paths) == 0:
    raise ValueError("❌ No images found! Check dataset path.")

# Dataset that returns L (1-ch) and AB (2-ch) normalized to [-1,1]
class ImageNetColorDataset(Dataset):
    def __init__(self, paths, size=(256,256)):
        self.paths = paths
        self.size = size
        self.toPIL = T.ToPILImage()
        self.transform_rgb = T.Compose([
            T.Resize(size),
            T.CenterCrop(size),
            T.ToTensor(),  # [0,1]
        ])
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        img_t = self.transform_rgb(img)  # [3,H,W], [0,1]
        # Convert to LAB (use skimage's rgb2lab expects HWC)
        img_np = np.transpose(img_t.numpy(), (1,2,0))  # HWC in [0,1]
        lab = rgb2lab(img_np).astype(np.float32)      # L in [0,100], A,B approx [-128,127]
        L = lab[:,:,0]  # H W
        AB = lab[:,:,1:3].transpose(2,0,1)  # 2,H,W
        # Normalize to training scheme:
        # L_norm = (L/50) - 1  => maps [0,100] -> [-1,1]
        L_norm = (L / 50.0) - 1.0
        # AB_norm = AB / 128  -> approx [-1,1]
        AB_norm = AB / 128.0
        L_t = torch.from_numpy(L_norm).unsqueeze(0).float()
        AB_t = torch.from_numpy(AB_norm).float()
        return {'L': L_t, 'AB': AB_t}

# create dataset and dataloader (you can use subset for quick debugging)
# Create dataset & dataloader
dataset = ImageNetColorDataset(paths, size=(256,256))
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                          num_workers=6, pin_memory=True)
print("✅ Dataset and DataLoader ready. Num samples:", len(dataset))

Found 536688 image files (first 3): ['/home/mmanani/ImageColourizationDataSet/ILSVRC/Data/DET/val/ILSVRC2012_val_00015406.JPEG', '/home/mmanani/ImageColourizationDataSet/ILSVRC/Data/DET/val/ILSVRC2012_val_00031533.JPEG', '/home/mmanani/ImageColourizationDataSet/ILSVRC/Data/DET/val/ILSVRC2013_val_00001830.JPEG']
✅ Dataset and DataLoader ready. Num samples: 536688


In [13]:
# Cell 4: Generator & Discriminator (use your versions if already defined)
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 2, 4, 2, 1), nn.Tanh()
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, 4, 1, 0), nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)


In [14]:
# Cell 5: Helpers for conversion + metrics
def lab_tensor_from_L_and_AB(L_tensor, AB_tensor):
    """Convert L (B,1,H,W) [-1,1] and AB (B,2,H,W) [-1,1] to numpy LAB HWC with L[0..100], AB[-128..127]."""
    L = L_tensor.detach().cpu().numpy()[0,0]   # H,W
    AB = AB_tensor.detach().cpu().numpy()[0]   # 2,H,W
    L_denorm = (L + 1.0) * 50.0
    AB_denorm = AB * 128.0
    lab = np.zeros((L_denorm.shape[0], L_denorm.shape[1], 3), dtype=np.float32)
    lab[:,:,0] = L_denorm
    lab[:,:,1:] = np.transpose(AB_denorm, (1,2,0))
    return lab

def lab_batch_to_rgb_tensor(L_batch, AB_batch):
    """Batch convert: inputs are torch tensors on device. Output: [B,3,H,W] in [-1,1]"""
    b = L_batch.size(0)
    rgb_list = []
    L_cpu = L_batch.detach().cpu()
    AB_cpu = AB_batch.detach().cpu()
    for i in range(b):
        lab = lab_tensor_from_L_and_AB(L_cpu[i:i+1], AB_cpu[i:i+1])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rgb = lab2rgb(lab)  # HWC in [0,1]
        #rgb = lab2rgb(lab)  # HWC in [0,1]
        rgb_t = torch.from_numpy(rgb.transpose(2,0,1)).float()  # CHW [0,1]
        rgb_t = (rgb_t * 2.0) - 1.0  # to [-1,1]
        rgb_list.append(rgb_t)
    rgb_batch = torch.stack(rgb_list, dim=0).to(device)
    return rgb_batch

def tensor_to_display_np(tensor):
    """Convert [B,3,H,W] (-1..1 or 0..1) -> single HWC [0,1] numpy for display"""
    t = tensor.detach().cpu()
    if t.dim()==4:
        t = t[0]
    if t.min() < 0:
        t = (t + 1.0) / 2.0
    t = torch.clamp(t, 0.0, 1.0)
    return t.permute(1,2,0).numpy()

def psnr_np(a,b):
    mse = np.mean((a - b)**2)
    if mse == 0:
        return float('inf')
    return 10 * math.log10(1.0 / (mse + 1e-10))

def ssim_np(a,b):
    try:
        return ssim_fn(a, b, multichannel=True, data_range=1.0)
    except Exception:
        return max(0.0, 1.0 - np.mean(np.abs(a-b)))


In [15]:
# Cell 6: instantiate & prepare training utilities
G = Generator().to(device)
D = Discriminator().to(device)

optimizer_G = optim.Adam(G.parameters(), lr=learning_rate_g, betas=betas)
optimizer_D = optim.Adam(D.parameters(), lr=learning_rate_d, betas=betas)

def lambda_lr(epoch):
    if epoch < decay_start_epoch:
        return 1.0
    return max(0.0, (num_epochs - epoch) / float(max(1, num_epochs - decay_start_epoch)))

scheduler_G = torch.optim.lr_scheduler.LambdaLR(optimizer_G, lr_lambda=lambda_lr)
scheduler_D = torch.optim.lr_scheduler.LambdaLR(optimizer_D, lr_lambda=lambda_lr)

bce = nn.BCELoss()
l1 = nn.L1Loss()
scaler = GradScaler()
print("Models, optimizers ready.")


Models, optimizers ready.


/tmp/ipykernel_7448/1674682007.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [16]:
# Cell 7: label generators, noise and CSV init
def sample_real_labels(shape, device=device):
    low, high = real_label_smooth_min, real_label_smooth_max
    return (low + (high - low) * torch.rand(shape, device=device))

def sample_fake_labels(shape, device=device):
    return (torch.rand(shape, device=device) * fake_label_max)

def add_input_noise(x, std=d_input_noise_std):
    if std <= 0:
        return x
    noise = torch.randn_like(x) * std
    return torch.clamp(x + noise, -1.0, 1.0)

# init CSV
with open(log_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch","batch_idx","G_loss","D_loss","PSNR","SSIM","lr_g","lr_d","time_elapsed_s"])
print("CSV logging initialized at", log_csv)


CSV logging initialized at /home/mmanani/mtechpracticals/semester-4/ChatGPT/training_log.csv


In [17]:
# Cell 8: Sanity test using a single batch
G.eval(); D.eval()
with torch.no_grad():
    sample = next(iter(train_loader))
    L = sample['L'].to(device)
    AB = sample['AB'].to(device)
    fake_AB = G(L)
    fake_rgb = lab_batch_to_rgb_tensor(L, fake_AB)
    disp = tensor_to_display_np(fake_rgb)
    plt.figure(figsize=(4,4)); plt.imshow(disp); plt.axis('off'); plt.title("Fake sample (sanity)")
print("Sanity check done.")


NameError: name 'warnings' is not defined

In [ ]:
# run once before training
try:
    del scaler
except Exception:
    pass
scaler = torch.cuda.amp.GradScaler()


/tmp/ipykernel_15765/1137870093.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
# ======= CELL 9: Complete Training Loop (revised - FP32, no AMP) =======
import time
import csv
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="skimage")

# Loss
bce_logits = nn.BCEWithLogitsLoss()

# Device
device_type = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device_type)

# Move models to device (if not already)
G.to(device)
D.to(device)

# Initialize loop counters if not present
try:
    start_time
except NameError:
    start_time = time.time()
global_step = 0

# training
num_batches = len(train_loader)
print(f"Starting training: epochs={num_epochs}, batches_per_epoch={num_batches}")

for epoch in range(1, num_epochs + 1):
    epoch_start = time.time()
    G.train(); D.train()

    for batch_idx, batch in enumerate(train_loader):
        # Unpack batch (keeps your variable names L and AB)
        L = batch['L'].to(device)      # [B,1,H,W], normalized as in dataset
        AB = batch['AB'].to(device)    # [B,2,H,W], normalized as in dataset
        b_size = L.size(0)

        # ----------------------------------------------------------------
        # 1) Train Discriminator (FP32)
        # ----------------------------------------------------------------
        optimizer_D.zero_grad()

        # Real RGB from ground truth AB + L
        real_rgb = lab_batch_to_rgb_tensor(L, AB)   # [B,3,H,W] in [-1,1] (as implemented earlier)
        real_in = add_input_noise(real_rgb, d_input_noise_std)
        pred_real = D(real_in)                       # shape depends on D (patches)

        # convert patch to scalar prediction per sample
        pred_real_mean = pred_real.view(b_size, -1).mean(1, keepdim=True)
        labels_real = sample_real_labels(pred_real_mean.shape, device=device)

        # If pred_real_mean are in (0,1) (sigmoid applied in D), convert to logits for BCEWithLogitsLoss:
        if (pred_real_mean.min().item() >= 0.0) and (pred_real_mean.max().item() <= 1.0):
            pred_real_logits = torch.logit(pred_real_mean.clamp(1e-6, 1.0 - 1e-6))
        else:
            pred_real_logits = pred_real_mean  # already logits

        loss_D_real = bce_logits(pred_real_logits, labels_real)

        # Fake images
        fake_AB = G(L)                               # generator output [B,2,H,W]
        fake_rgb = lab_batch_to_rgb_tensor(L, fake_AB)  # [B,3,H,W] in [-1,1]
        fake_in = add_input_noise(fake_rgb.detach(), d_input_noise_std)
        pred_fake = D(fake_in)
        pred_fake_mean = pred_fake.view(b_size, -1).mean(1, keepdim=True)
        labels_fake = sample_fake_labels(pred_fake_mean.shape, device=device)

        if (pred_fake_mean.min().item() >= 0.0) and (pred_fake_mean.max().item() <= 1.0):
            pred_fake_logits = torch.logit(pred_fake_mean.clamp(1e-6, 1.0 - 1e-6))
        else:
            pred_fake_logits = pred_fake_mean

        loss_D_fake = bce_logits(pred_fake_logits, labels_fake)

        # Combine D loss (mean of real+fake is optional; keep sum for gradient strength)
        loss_D = 0.5 * (loss_D_real + loss_D_fake)

        # backward + step for D (FP32)
        loss_D.backward()
        torch.nn.utils.clip_grad_norm_(D.parameters(), 5.0)
        optimizer_D.step()

        # ----------------------------------------------------------------
        # 2) Train Generator (FP32)
        # ----------------------------------------------------------------
        optimizer_G.zero_grad()

        fake_AB = G(L)
        fake_rgb = lab_batch_to_rgb_tensor(L, fake_AB)   # [B,3,H,W]
        pred_fake_for_g = D(fake_rgb)
        pred_fake_mean_g = pred_fake_for_g.view(b_size, -1).mean(1, keepdim=True)

        # If D applied Sigmoid inside, convert to logits for BCEWithLogitsLoss
        if (pred_fake_mean_g.min().item() >= 0.0) and (pred_fake_mean_g.max().item() <= 1.0):
            pred_fake_logits_g = torch.logit(pred_fake_mean_g.clamp(1e-6, 1.0 - 1e-6))
        else:
            pred_fake_logits_g = pred_fake_mean_g

        # Generator tries to make D predict 'real' (smoothed)
        target_for_g = sample_real_labels(pred_fake_logits_g.shape, device=device)
        loss_G_adv = bce_logits(pred_fake_logits_g, target_for_g)

        # Pixel-wise loss (RGB L1) using real RGB
        real_rgb_for_loss = lab_batch_to_rgb_tensor(L, AB)
        loss_pixel = nn.L1Loss()(fake_rgb, real_rgb_for_loss)

        # total generator loss
        loss_G = loss_G_adv + lambda_pixel * loss_pixel

        # backward + step G (FP32)
        loss_G.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 5.0)
        optimizer_G.step()

        # ----------------------------------------------------------------
        # 3) Metrics, logging, and periodic samples
        # ----------------------------------------------------------------
        with torch.no_grad():
            real_vis = tensor_to_display_np(real_rgb_for_loss[0:1])   # [0,1] HWC
            fake_vis = tensor_to_display_np(fake_rgb[0:1])           # [0,1] HWC
            psnr_val = psnr_np(real_vis, fake_vis)
            ssim_val = ssim_np(real_vis, fake_vis)

        elapsed = time.time() - start_time
        # Append to CSV log
        with open(log_csv, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([epoch, batch_idx, float(loss_G.item()), float(loss_D.item()), float(psnr_val), float(ssim_val), optimizer_G.param_groups[0]['lr'], optimizer_D.param_groups[0]['lr'], elapsed])

        # print progress every 100 iters
        if global_step % 100 == 0:
            print(f"Epoch[{epoch}/{num_epochs}] Batch[{batch_idx+1}/{num_batches}] G_loss={loss_G.item():.4f} D_loss={loss_D.item():.4f} PSNR={psnr_val:.3f} SSIM={ssim_val:.3f}")

        # save sample grid periodically
        if global_step % sample_every_batches == 0:
            fname = sample_dir / f"ep{epoch:03d}_step{global_step:06d}.png"
            grid = np.concatenate([real_vis, fake_vis], axis=1)
            plt.imsave(fname, grid)

        global_step += 1

    # End of epoch: step schedulers and save checkpoint
    scheduler_G.step(); scheduler_D.step()

    ckpt = {
        "epoch": epoch,
        "G_state_dict": G.state_dict(),
        "D_state_dict": D.state_dict(),
        "optimizer_G": optimizer_G.state_dict(),
        "optimizer_D": optimizer_D.state_dict(),
        "scheduler_G": scheduler_G.state_dict(),
        "scheduler_D": scheduler_D.state_dict()
    }
    torch.save(ckpt, checkpoint_path)
    print(f"Epoch {epoch} finished. Time: {time.time()-epoch_start:.1f}s LR_G={optimizer_G.param_groups[0]['lr']:.2e} LR_D={optimizer_D.param_groups[0]['lr']:.2e}")

print("Training loop finished.")


Starting training: epochs=100, batches_per_epoch=33543
Epoch[1/100] Batch[1/33543] G_loss=30.3988 D_loss=0.6916 PSNR=7.909 SSIM=0.669
Epoch[1/100] Batch[101/33543] G_loss=30.8534 D_loss=0.4161 PSNR=8.428 SSIM=0.705
Epoch[1/100] Batch[201/33543] G_loss=29.3647 D_loss=0.3677 PSNR=7.658 SSIM=0.657
Epoch[1/100] Batch[301/33543] G_loss=31.0921 D_loss=0.3685 PSNR=8.789 SSIM=0.721
Epoch[1/100] Batch[401/33543] G_loss=30.9772 D_loss=0.3444 PSNR=8.744 SSIM=0.713
Epoch[1/100] Batch[501/33543] G_loss=31.0260 D_loss=0.3583 PSNR=8.769 SSIM=0.694
Epoch[1/100] Batch[601/33543] G_loss=32.4624 D_loss=0.3540 PSNR=8.549 SSIM=0.691
Epoch[1/100] Batch[701/33543] G_loss=31.1124 D_loss=0.3411 PSNR=9.609 SSIM=0.761
Epoch[1/100] Batch[801/33543] G_loss=30.8737 D_loss=0.3772 PSNR=7.989 SSIM=0.686
Epoch[1/100] Batch[901/33543] G_loss=30.2168 D_loss=0.3141 PSNR=8.916 SSIM=0.725
Epoch[1/100] Batch[1001/33543] G_loss=31.1498 D_loss=0.3762 PSNR=8.759 SSIM=0.730
Epoch[1/100] Batch[1101/33543] G_loss=31.3067 D_loss=0.

KeyboardInterrupt: 

In [ ]:
# Cell 10: Plot aggregated epoch statistics
import pandas as pd
log = pd.read_csv(log_csv)
epoch_stats = log.groupby('epoch').mean().reset_index()

plt.figure(figsize=(14,4))
plt.subplot(1,3,1)
plt.plot(epoch_stats['epoch'], epoch_stats['G_loss'], label='G_loss')
plt.plot(epoch_stats['epoch'], epoch_stats['D_loss'], label='D_loss')
plt.xlabel('epoch'); plt.legend(); plt.title('Losses')

plt.subplot(1,3,2)
plt.plot(epoch_stats['epoch'], epoch_stats['PSNR'])
plt.xlabel('epoch'); plt.title('PSNR')

plt.subplot(1,3,3)
plt.plot(epoch_stats['epoch'], epoch_stats['SSIM'])
plt.xlabel('epoch'); plt.title('SSIM')

plt.tight_layout(); plt.show()


In [ ]:
# Cell 11: Inference - colorize a given grayscale image using trained generator
def colorize_image_path(img_path, checkpoint=checkpoint_path):
    # load checkpoint if needed
    ckpt = torch.load(checkpoint, map_location=device)
    G.load_state_dict(ckpt['G_state_dict'])
    G.to(device); G.eval()

    img = Image.open(img_path).convert("L")
    # resize same as training
    img = img.resize((256,256))
    L = np.asarray(img).astype(np.float32)  # H W in [0,255]
    L = (L / 255.0) * 100.0                 # convert to [0,100] like LAB L
    L_norm = (L / 50.0) - 1.0               # normalize to [-1,1]
    L_tensor = torch.from_numpy(L_norm).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        AB_pred = G(L_tensor)
        rgb_pred = lab_batch_to_rgb_tensor(L_tensor, AB_pred)
        disp = tensor_to_display_np(rgb_pred)
    plt.figure(figsize=(6,6)); plt.imshow(disp); plt.axis('off'); plt.title("Colorized")
